# MGS-7c : Rosenbrock (vallée étroite) + Griewank (produit-cosinus) en projection N-D

[← MGS-7b LandscapeMultidim](MGS-07b-LandscapeMultidim.ipynb) · [MGS-7d MichalewiczDixonPrice →](MGS-07d-MichalewiczDixonPrice.ipynb)

## Pourquoi Rosenbrock et Griewank ?

MGS-7b a couvert les **quatre fonctions** classiques du banc CEC :
**Sphère**, **Rastrigin**, **Schwefel**, **Ackley**.

Restent deux fonctions au comportement **structurellement différent** que la projection N-D
doit affronter :

1. **Rosenbrock** $f(\vec x) = -\sum_{i=1}^{n-1} \left(100(x_{i+1} - x_i^2)^2 + (1-x_i)^2\right)$ —
   la *vallée-courbe-étroite banane*. L'optimum est dans $x_i \approx 1$, mais la trajectoire
   de descente est piégée dans un couloir de largeur $\sim 10^{-2}$. Recommandé $[-2.048, 2.048]$.

2. **Griewank** $f(\vec x) = \frac{1}{4000}\sum_i x_i^2 - \prod_i \cos\left(\frac{x_i}{\sqrt{i}}\right) + 1$ —
   le *produit-cosinus-haute-fréquence*. Contrairement à **Ackley** (somme de cosinus), Griewank
   multiplie des cosinus à fréquences **décroissantes** : $\cos(x_0)$, $\cos(x_1/\sqrt{2})$,
   $\cos(x_2/\sqrt{3})$, etc. Échelle large $[-600, 600]$.

## Ce que ce notebook teste

**Question pédagogique** : la projection MAX (cf MGS-7b cellule `RenderHeatmap` N-D, port #7483)
gomme-t-elle les structures fines de Rosenbrock (vallée 1-pixel-large) **plus vite** que
celles d'Ackley (bassine large multi-pixels) ? Et comment **Griewank** se distingue-t-il
de **Ackley** en 2-D alors que les deux reposent sur du cosinus ?

**Limite intrinsèque documentée** : si la heatmap Griewank-n=30 devient quasi-uniforme,
c'est une **signature de la projection MAX** (le produit de cosinus à fréquences décroissantes
s'aplatit sous MAX), pas un bug de la projection.

## Reproductibilité

Les seeds utilisées (42, 7, 99) sont explicites dans les cellules. Les heatmaps sont
générées via le moteur `KnownFunctionLandscape.RenderHeatmap` du submodule
`MetaGeneticSharp.Extensions` (port PR #7583 + #7776).

In [1]:
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Extensions/bin/Debug/net9.0/MetaGeneticSharp.Extensions.dll"
// SkiaSharp (rendu PNG par ToPng) vient de NuGet : le kernel cable alors le binaire natif de la
// plateforme (Linux par SkiaSharp.NativeAssets.Linux.NoDependencies), ce que ne fait pas un #r
// par chemin. Version = reference du fork.
#r "nuget: SkiaSharp, 3.119.4"
#r "nuget: SkiaSharp.NativeAssets.Linux.NoDependencies, 3.119.4"

using MetaGeneticSharp;
using GeneticSharp;

// Verifier l'API du submodule -- check BLOQUANT. Le porteur des bornes est
// KnownFunctionsBounds.For (KnownFunctions.cs), unique point d'acces canonique.
var boundsFor = typeof(KnownFunctionsBounds).GetMethod("For",
    System.Reflection.BindingFlags.Public | System.Reflection.BindingFlags.Static);
if (boundsFor is null)
    throw new InvalidOperationException(
        "KnownFunctionsBounds.For absent du submodule -- recompiler MetaGeneticSharp.Extensions");
var rosenbrockType = System.Type.GetType("MetaGeneticSharp.RosenbrockFitness, MetaGeneticSharp.Extensions");
var griewankType = System.Type.GetType("MetaGeneticSharp.GriewankFitness, MetaGeneticSharp.Extensions");
if (rosenbrockType is null || griewankType is null)
    throw new InvalidOperationException(
        "Classes de fitness introuvables dans MetaGeneticSharp.Extensions");
Console.WriteLine($"KnownFunctionsBounds.For present: {boundsFor is not null}");
Console.WriteLine($"RosenbrockFitness charge: {rosenbrockType.FullName}");
Console.WriteLine($"GriewankFitness charge: {griewankType.FullName}");

// Affichage des bornes en culture invariante (separateur decimal point,
// independant des reglages de la machine hote).
var fmt = "0.###";
var fmt4 = "0.####";
var invCulture = System.Globalization.CultureInfo.InvariantCulture;
// Echelles recommandees : mesurees par l'appel REEL au submodule (KnownFunctions.cs:236),
// pas recopiees en litteral.
var (rosenbrockMin, rosenbrockMax) = KnownFunctionsBounds.For(rosenbrockType);
var (griewankMin, griewankMax) = KnownFunctionsBounds.For(griewankType);
Console.WriteLine($"Rosenbrock: range [{rosenbrockMin.ToString(fmt, invCulture)}, {rosenbrockMax.ToString(fmt, invCulture)}] (KnownFunctionsBounds.For)");
Console.WriteLine($"Griewank:   range [{griewankMin.ToString(fmt, invCulture)}, {griewankMax.ToString(fmt, invCulture)}] (KnownFunctionsBounds.For)");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages SkiaSharp, 3.119.4 SkiaSharp.NativeAssets.Linux.NoDependencies, 3.119.4

Loading extensions from `/root/.nuget/packages/skiasharp/3.119.4/interactive-extensions/dotnet/SkiaSharp.DotNet.Interactive.dll`

KnownFunctionsBounds.For present: True


RosenbrockFitness charge: MetaGeneticSharp.RosenbrockFitness


GriewankFitness charge: MetaGeneticSharp.GriewankFitness


Rosenbrock: range [-2.048, 2.048] (KnownFunctionsBounds.For)


Griewank:   range [-600, 600] (KnownFunctionsBounds.For)


## Rosenbrock en $n = 2, 5, 30$ : la vallée diagonale disparaît-elle ?

**En 2-D**, Rosenbrock dessine une *vallée étroite* en banane qui serpente du coin
supérieur-gauche $(-2, 4)$ environ jusqu'au point optimal $(1, 1)$.
La heatmap est **asymétrique** : le coin $(x_1, x_2) = (-2, 4)$ est plus rouge
(vallée en pente descendante raide) que le coin symétrique $(4, -2)$.

**En 30-D**, chaque pixel $(x_1, x_2)$ doit explorer 28 dimensions cachées via
MAX-projection. Comme la vallée est **étroite** (largeur $\sim 10^{-2}$), un échantillon
aléatoire tombe rarement *dans* la vallée : le raisonnement attendu est que la MAX-projection
retient alors surtout une fitness *hors vallée*, et que la heatmap s'uniformise.
La section « Mesure du rendu visuel » teste cette attente par la variance de chaque canal.
L'hypothèse d'une vulnérabilité plus forte de Rosenbrock qu'Ackley à la projection MAX
n'est pas testée dans ce carnet, qui ne mesure que Rosenbrock et Griewank.

In [2]:
using System.IO;
using System.Security.Cryptography;
// Piege #12457 : un unique Random etatique partage entre les dimensions rend
// d30 quasi-confondu a d2. RNG independant par dimension + assertion de distinction.
var written = new List<(int dim, string path, string hash)>();
foreach (int dim in new[] { 2, 5, 30 })
{
    using var h = KnownFunctionLandscape.RenderHeatmap(
        new RosenbrockFitness(), dimension: dim, nbSamples: 50, rng: new Random(42 + dim),
        width: 120, height: 120);

    var path = "assets/landscape_multidim_rosenbrock_d" + dim + ".png";
    var bytes = h.ToPng();
    File.WriteAllBytes(path, bytes);

    var hash = Convert.ToHexString(SHA256.HashData(bytes));
    written.Add((dim, path, hash));

    var (px, py) = h.ToPixel(0.0, 0.0);
    Console.WriteLine("Rosenbrock dim=" + dim.ToString().PadLeft(2) + " - pixel(0,0) = " + h.Bitmap.GetPixel(px, py) + " (PNG: " + path + ", sha256=" + hash.Substring(0, 12) + ").");
}
if (written.Select(x => x.hash).Distinct().Count() != written.Count)
{
    foreach (var w in written) Console.Error.WriteLine("  d" + w.dim + "  " + w.hash);
    throw new InvalidOperationException("Collision de rendu Rosenbrock : deux dimensions ont produit le meme PNG (hash identiques).");
}
Console.WriteLine("Rosenbrock d2/d5/d30 distincts : "
    + string.Join(", ", written.Select(w => "d" + w.dim + "=" + w.hash.Substring(0, 12))));

Rosenbrock dim= 2 - pixel(0,0) = Color [A=255, R=255, G=0, B=0] (PNG: assets/landscape_multidim_rosenbrock_d2.png, sha256=0D7500C2FDD7).


Rosenbrock dim= 5 - pixel(0,0) = Color [A=255, R=255, G=7, B=0] (PNG: assets/landscape_multidim_rosenbrock_d5.png, sha256=846C609BCA1D).


Rosenbrock dim=30 - pixel(0,0) = Color [A=255, R=234, G=255, B=0] (PNG: assets/landscape_multidim_rosenbrock_d30.png, sha256=C25AA745E8C3).


Rosenbrock d2/d5/d30 distincts : d2=0D7500C2FDD7, d5=846C609BCA1D, d30=C25AA745E8C3



warning CS1701: En supposant que la référence d'assembly 'System.Drawing.Primitives, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'MetaGeneticSharp.Infrastructure' correspond à l'identité 'System.Drawing.Primitives, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Drawing.Primitives', il se peut que vous deviez fournir une stratégie runtime



## Griewank en $n = 2, 5, 30$ : produit-cosinus vs Ackley somme-cosinus

**Différence structurelle avec Ackley** (cf MGS-7b) :

| Fonction | Formule cosinus | Type | Signature 2-D |
|----------|----------------|------|---------------|
| **Ackley** | $-\exp(\frac{1}{n}\sum \cos(2\pi x_i))$ | **somme** | bassine large, anneaux concentriques |
| **Griewank** | $-\prod \cos(\frac{x_i}{\sqrt{i}})$ | **produit** | bandes ondulantes perpendiculaires |

En 2-D, le produit $\cos(x_0) \cdot \cos(x_1/\sqrt{2})$ génère des **bandes
verticales ondulantes** (la fréquence décroît avec l'index $i$, donc $\cos(x_1/\sqrt{2})$
est $\sqrt{2}$ fois plus lent que $\cos(x_0)$). En 30-D, le produit de 30 cosinus à fréquences très diverses doit s'aplatir
**presque entièrement** sous MAX : c'est l'attente que la section « Mesure du rendu visuel »
teste sur le canal vert.

**Limite intrinsèque** : si cette attente se confirme, Griewank en 30-D est peu lisible via
MAX-projection. Ce carnet ne mesure que $n = 2$, $5$ et $30$ : le seuil de dimension à partir
duquel cette perte apparaît n'est pas établi ici. C'est cohérent avec le fait que Griewank est
conçu pour tester les **métaheuristiques** (qui peuvent *descendre* le long de la structure),
pas les projections 2-D.

In [3]:
using System.IO;
using System.Security.Cryptography;
// Piege #12457 : un unique Random etatique partage entre les dimensions rend
// d30 quasi-confondu a d2. RNG independant par dimension + assertion de distinction.
var written = new List<(int dim, string path, string hash)>();
foreach (int dim in new[] { 2, 5, 30 })
{
    using var h = KnownFunctionLandscape.RenderHeatmap(
        new GriewankFitness(), dimension: dim, nbSamples: 50, rng: new Random(7 + dim),
        width: 120, height: 120);

    var path = "assets/landscape_multidim_griewank_d" + dim + ".png";
    var bytes = h.ToPng();
    File.WriteAllBytes(path, bytes);

    var hash = Convert.ToHexString(SHA256.HashData(bytes));
    written.Add((dim, path, hash));

    var (px, py) = h.ToPixel(0.0, 0.0);
    Console.WriteLine("Griewank dim=" + dim.ToString().PadLeft(2) + " - pixel(0,0) = " + h.Bitmap.GetPixel(px, py) + " (PNG: " + path + ", sha256=" + hash.Substring(0, 12) + ").");
}
if (written.Select(x => x.hash).Distinct().Count() != written.Count)
{
    foreach (var w in written) Console.Error.WriteLine("  d" + w.dim + "  " + w.hash);
    throw new InvalidOperationException("Collision de rendu Griewank : deux dimensions ont produit le meme PNG (hash identiques).");
}
Console.WriteLine("Griewank d2/d5/d30 distincts : "
    + string.Join(", ", written.Select(w => "d" + w.dim + "=" + w.hash.Substring(0, 12))));

Griewank dim= 2 - pixel(0,0) = Color [A=255, R=255, G=4, B=0] (PNG: assets/landscape_multidim_griewank_d2.png, sha256=BBB43EBED5F6).


Griewank dim= 5 - pixel(0,0) = Color [A=255, R=255, G=11, B=0] (PNG: assets/landscape_multidim_griewank_d5.png, sha256=7713E4022D7B).


Griewank dim=30 - pixel(0,0) = Color [A=255, R=42, G=255, B=0] (PNG: assets/landscape_multidim_griewank_d30.png, sha256=EC61F1CBDFD1).


Griewank d2/d5/d30 distincts : d2=BBB43EBED5F6, d5=7713E4022D7B, d30=EC61F1CBDFD1



warning CS1701: En supposant que la référence d'assembly 'System.Drawing.Primitives, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'MetaGeneticSharp.Infrastructure' correspond à l'identité 'System.Drawing.Primitives, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Drawing.Primitives', il se peut que vous deviez fournir une stratégie runtime



In [4]:
// Mesure de la structure par canal des six rendus ecrits ci-dessus.
// Les quantites du tableau « Tell visuel (mesure) » ci-dessous (variance et
// moyenne de R, G et B) sont PRODUITES ici au lieu d'etre recopiees dans la
// prose. Elles sont lues sur les PNG committes -- donc exactement sur les
// heatmaps des sections precedentes (graines 42 + dim pour Rosenbrock,
// 7 + dim pour Griewank), aucun re-rendu.
using System.IO;

var fichiers = new[] {
    ("Rosenbrock", "rosenbrock",  2), ("Rosenbrock", "rosenbrock",  5),
    ("Rosenbrock", "rosenbrock", 30),
    ("Griewank",   "griewank",    2), ("Griewank",   "griewank",    5),
    ("Griewank",   "griewank",   30),
};
var mesures = new List<(string fonction, int dim, double varR, double varG, double varB,
                       double moyR, double moyG, double moyB)>();
foreach (var (fonction, slug, dim) in fichiers)
{
    using var png = File.OpenRead("assets/landscape_multidim_" + slug + "_d" + dim + ".png");
    using var img = LandscapeMaps.DecodeBitmap(png);   // DirectBitmap, decodage SkiaSharp sans GDI+
    double sR = 0, sG = 0, sB = 0, qR = 0, qG = 0, qB = 0;
    long n = 0;
    for (int ligne = 0; ligne < img.Height; ligne++)
    {
        for (int colonne = 0; colonne < img.Width; colonne++)
        {
            var c = img.GetPixel(colonne, ligne);
            sR += c.R; sG += c.G; sB += c.B;
            qR += (double)c.R * c.R; qG += (double)c.G * c.G; qB += (double)c.B * c.B;
            n++;
        }
    }
    double mR = sR / n, mG = sG / n, mB = sB / n;
    mesures.Add((fonction, dim, qR / n - mR * mR, qG / n - mG * mG, qB / n - mB * mB,
                 mR, mG, mB));
}

Console.WriteLine("Structure par canal des six rendus (variance et moyenne de R, G, B) :");
foreach (var m in mesures)
{
    Console.WriteLine(m.fonction.PadRight(12) + " d" + m.dim.ToString().PadLeft(2)
        + "  var(R)=" + m.varR.ToString("0.0").PadLeft(8)
        + " var(G)=" + m.varG.ToString("0.0").PadLeft(8)
        + " var(B)=" + m.varB.ToString("0.0").PadLeft(8)
        + "  | moy(R)=" + m.moyR.ToString("0.0").PadLeft(6)
        + " moy(G)=" + m.moyG.ToString("0.0").PadLeft(6)
        + " moy(B)=" + m.moyB.ToString("0.0").PadLeft(6));
}


Structure par canal des six rendus (variance et moyenne de R, G, B) :


Rosenbrock   d 2  var(R)=  2736,8 var(G)=  7607,8 var(B)=   250,4  | moy(R)= 239,2 moy(G)=  81,4 moy(B)=   1,9


Rosenbrock   d 5  var(R)=  2070,8 var(G)=  6982,0 var(B)=   151,8  | moy(R)= 242,5 moy(G)=  96,3 moy(B)=   1,2


Rosenbrock   d30  var(R)=  5162,4 var(G)=   841,8 var(B)=   119,3  | moy(R)= 177,9 moy(G)= 243,5 moy(B)=   1,4


Griewank     d 2  var(R)=  7711,3 var(G)=  6692,0 var(B)=   832,7  | moy(R)= 189,9 moy(G)= 190,7 moy(B)=   6,8


Griewank     d 5  var(R)=  6424,1 var(G)=  5591,2 var(B)=   321,7  | moy(R)= 199,2 moy(G)= 195,0 moy(B)=   3,3


Griewank     d30  var(R)=  5084,6 var(G)=    69,8 var(B)=  1038,1  | moy(R)=  71,8 moy(G)= 254,0 moy(B)=  14,0



warning CS1701: En supposant que la référence d'assembly 'System.Drawing.Primitives, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'MetaGeneticSharp.Infrastructure' correspond à l'identité 'System.Drawing.Primitives, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Drawing.Primitives', il se peut que vous deviez fournir une stratégie runtime



## Mesure du rendu visuel

Les six images produites par les cellules de rendu (`assets/landscape_multidim_{rosenbrock,griewank}_d{2,5,30}.png`) sont mesurées par variance et moyenne par canal (RGB). Les 36 valeurs du tableau sont **recopiées de la sortie de la cellule de mesure** ci-dessus, calculée sur les 14 400 pixels de chaque rendu, avec la variance de **population** (`ddof=0`, moyenne des carrés des écarts à la moyenne : la convention que reprend l'exercice 1). La cellule imprime ces mêmes valeurs avec la virgule décimale : `2736,8` y vaut `2736.8` ici.

| fonction | d | var(R) | var(G) | var(B) | moy(R) | moy(G) | moy(B) |
|---|---|---|---|---|---|---|---|
| Rosenbrock | 2 | 2736.8 | 7607.8 | 250.4 | 239.2 | 81.4 | 1.9 |
| Rosenbrock | 5 | 2070.8 | 6982.0 | 151.8 | 242.5 | 96.3 | 1.2 |
| Rosenbrock | 30 | 5162.4 | 841.8 | 119.3 | 177.9 | 243.5 | 1.4 |
| Griewank | 2 | 7711.3 | 6692.0 | 832.7 | 189.9 | 190.7 | 6.8 |
| Griewank | 5 | 6424.1 | 5591.2 | 321.7 | 199.2 | 195.0 | 3.3 |
| Griewank | 30 | 5084.6 | 69.8 | 1038.1 | 71.8 | 254.0 | 14.0 |

**Constat mesuré sur les six rendus** :

1. **Rosenbrock** : la variance du canal R est plus élevée en dimension 30 (5162,4) qu'en dimension 2 (2736,8). La structure fine ne disparaît donc pas dans ce rendu. Une piste d'explication est le relief inter-pixels laissé par la MAX-projection avec `nbSamples: 50` ; le carnet ne la teste pas.
2. **Rosenbrock en 30-D** : la couleur est un champ orange-vert (moy(R)=177,9, moy(G)=243,5) où le bleu disparaît (moy(B)=1,4).
3. **Rosenbrock en 2-D et 5-D** : le gradient de rouge domine (moy(R) 239,2 et 242,5, contre moy(G) 81,4 et 96,3). **Griewank en 30-D** : le canal vert est quasi uniforme (moy(G)=254,0, var(G)=69,8), avec un rouge résiduel (moy(R)=71,8).

**Point de méthode** : chaque dimension reçoit son propre générateur aléatoire (graine 42 + dimension pour Rosenbrock, 7 + dimension pour Griewank). Un générateur unique, partagé entre les dimensions, est un piège : il fait ressembler le rendu de dimension 30 à celui de dimension 2. La cellule de rendu vérifie donc que les trois empreintes SHA-256 sont distinctes. Le rendu est conçu pour être déterministe par pixel (graine dérivée du pixel, indépendante de l'ordonnancement parallèle).

## Exercices

### Exercice 1 : quantifier la *disparition* de la vallée Rosenbrock

Mesurez la **variance spatiale** du canal rouge entre les heatmaps Rosenbrock en dimensions 2, 5 et 30. Si la vallée disparaît, cette variance devrait chuter nettement — mais le rendu déterministe ci-dessus suggère au contraire une hausse en dimension 30. Calculez les trois variances puis le ratio `variance(d2) / variance(d30)` pour confronter l'intuition à la mesure.

**Indice :** collectez la composante `R` des 14 400 pixels de chaque bitmap, puis utilisez la moyenne des carrés des écarts à la moyenne.

In [5]:
// EXERCICE 1 : variance du canal rouge selon la dimension.
// TODO etudiant : calculer la variance de R pour chaque heatmap.
var variancesRouges = new List<(int dimension, double variance)>();
foreach (int dim in new[] { 2, 5, 30 })
{
    using var heatmap = KnownFunctionLandscape.RenderHeatmap(
        new RosenbrockFitness(), dimension: dim, nbSamples: 50,
        rng: new Random(42 + dim), width: 120, height: 120);

    // Etape 1 : collecter heatmap.Bitmap.GetPixel(x, y).R pour tous les pixels.
    // Etape 2 : calculer moyenne puis moyenne des ecarts au carre.
    // Etape 3 : ajouter (dim, variance) a variancesRouges.
}

Console.WriteLine("Exercice 1 a completer : mesurer variance(R) en d2, d5 et d30.");

Exercice 1 a completer : mesurer variance(R) en d2, d5 et d30.


### Exercice 2 : comparer les signatures fréquentielles de Griewank et Ackley

Comparez en dimension 2 la structure répétitive de **Griewank** à celle d'**Ackley**. À partir du canal rouge de chaque heatmap, construisez un profil horizontal moyen, puis comparez la fréquence dominante des deux profils. L'objectif est de vérifier quantitativement la différence entre produit de cosinus et somme de cosinus.

**Indice :** moyennez le canal rouge colonne par colonne avant d'appliquer une transformée discrète de Fourier ; ignorez la composante continue lors de la recherche du maximum spectral.

In [6]:
// EXERCICE 2 : profils spectraux Griewank et Ackley en dimension 2.
// TODO etudiant : extraire les profils horizontaux moyens puis leur frequence dominante.
var heatmapGriewank = KnownFunctionLandscape.RenderHeatmap(
    new GriewankFitness(), dimension: 2, nbSamples: 50,
    rng: new Random(7), width: 120, height: 120);
var heatmapAckley = KnownFunctionLandscape.RenderHeatmap(
    new AckleyFitness(), dimension: 2, nbSamples: 50,
    rng: new Random(99), width: 120, height: 120);

// Etape 1 : moyenner le canal R de chaque colonne pour chaque heatmap.
// Etape 2 : calculer la transformee discrete de Fourier des deux profils.
// Etape 3 : ignorer la composante continue et comparer les pics dominants.
Console.WriteLine("Exercice 2 a completer : comparer les frequences dominantes Griewank/Ackley.");
heatmapGriewank.Dispose();
heatmapAckley.Dispose();

Exercice 2 a completer : comparer les frequences dominantes Griewank/Ackley.


### Exercice 3 : compter les minima locaux en 2-D

Estimez la multimodalité visible en comptant les **minima locaux stricts** dans les heatmaps Rosenbrock et Griewank en dimension 2. Un pixel intérieur est un minimum local si son canal rouge est strictement inférieur à ceux de ses quatre voisins directs. Comparez les deux comptes et affichez leur ratio.

**Indice :** parcourez seulement les pixels intérieurs (`1..118`) afin que chaque pixel possède quatre voisins ; traitez séparément un éventuel dénominateur nul.

In [7]:
// EXERCICE 3 : minima locaux stricts en dimension 2.
// TODO etudiant : compter les minima locaux de Rosenbrock et Griewank.
var heatmapRosenbrock2D = KnownFunctionLandscape.RenderHeatmap(
    new RosenbrockFitness(), dimension: 2, nbSamples: 50,
    rng: new Random(42), width: 120, height: 120);
var heatmapGriewank2D = KnownFunctionLandscape.RenderHeatmap(
    new GriewankFitness(), dimension: 2, nbSamples: 50,
    rng: new Random(42), width: 120, height: 120);

// Etape 1 : parcourir x et y de 1 a 118.
// Etape 2 : comparer R(x,y) aux quatre voisins directs.
// Etape 3 : afficher les deux comptes et leur ratio.
Console.WriteLine("Exercice 3 a completer : compter les minima locaux Rosenbrock/Griewank.");
heatmapRosenbrock2D.Dispose();
heatmapGriewank2D.Dispose();

Exercice 3 a completer : compter les minima locaux Rosenbrock/Griewank.


## Liens

- [MGS-7b LandscapeMultidim](MGS-07b-LandscapeMultidim.ipynb) — Ackley dim ∈ {2, 30}, Rastrigin/Schwefel/Sphere dim ∈ {2, 5, 10, 30}.
- [MGS-6 Benchmarks](MGS-06-Benchmarks.ipynb) — les 10 fonctions canoniques et leur banc CEC d'origine.
- [MGS-8 LandscapeExplorer](MGS-08-LandscapeExplorer.ipynb) — exploration interactive GTK#.
- Issue [#7483](https://github.com/jsboige/CoursIA/issues/7483) — port de la projection N-D (parent).
- Issue [#7997](https://github.com/jsboige/CoursIA/issues/7997) — MGS-7c Rosenbrock+Griewank (sujet de ce notebook).
- PR [#7583](https://github.com/jsboige/CoursIA/pull/7583) — port initial MGS-7b N-D.
- PR [#7993](https://github.com/jsboige/CoursIA/pull/7993) — MGS-7b Ackley.
- Inversion chromatique chiffrable de d2 à d30 : signature de la projection MAX.